# 03_evaluate (推理加速修复版)

Span-level 评估 harness,针对推理速度问题做了 4 个关键修复:

1. **`padding_side='left'`**:decoder LLM 推理必须左 padding,否则 warning + 速度暴跌 + 结果不稳
2. **`BATCH_SIZE=8`**:推理不做 backward,显存宽裕,从 4 加到 8 翻倍吞吐
3. **`MAX_NEW_TOKENS=1500`**:原值 2048 过大,span-tagged 输出最多比 input 长 20-30%,1500 够用
4. **去掉无效的 `temperature/top_p/top_k`**:greedy 模式下这些参数被忽略且产生 warning

预期速度:**2-5 samples/s**,3800 条样本约 **15-30 分钟** 跑完(vs 修复前的 50 小时)。

## 使用前确认

- 训练完成,`../outputs/qwen3_4b_pii_lora_tagged/` 有 `adapter_model.safetensors`
- `../data/processed/test_ground_truth.jsonl` 存在(01 notebook 的输出)
- `redaction_utils.py` 和本 notebook 同目录

In [1]:
import json
import os
import sys
import time
from collections import defaultdict, Counter
from typing import List, Dict, Any, Tuple

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

sys.path.insert(0, ".")
from redaction_utils import parse_annotated, apply_redaction

/home/admin/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 配置

In [2]:
BASE_MODEL   = "../../model/Qwen3-4B-Instruct-2507"
ADAPTER_DIR  = "../outputs/qwen3_4b_pii_lora_tagged"   # ← 对齐 02 notebook 的 OUTPUT_DIR
USE_MERGED   = False
MERGED_DIR   = "../outputs/qwen3_4b_pii_merged_tagged"

TEST_GT_PATH = "../data/processed/test_ground_truth.jsonl"
META_PATH    = "../data/processed/meta.json"
REPORT_DIR   = "../outputs/eval_report"

# ======= 推理参数(修复版) =======
BATCH_SIZE     = 8       # 从 4 改成 8
MAX_NEW_TOKENS = 1500    # 从 2048 改成 1500(span-tagged 输出最多比 input 长 30%)
MAX_INPUT_LEN  = 1536    # tokenizer 截断长度

# ======= 评估参数 =======
PARTIAL_IOU_THRESHOLD = 0.5

HIGH_RISK_TYPES = {
    "AU_TFN", "AU_PASSPORT", "AU_DRIVERS_LICENCE",
    "MEDICARE_NUMBER", "PAYMENT_CARD_NUMBER", "AU_BANK_ACCOUNT",
}
HIGH_RISK_WEIGHT = 3.0
LOW_RISK_WEIGHT  = 1.0

os.makedirs(REPORT_DIR, exist_ok=True)
assert os.path.exists(TEST_GT_PATH), f"找不到 {TEST_GT_PATH}"
assert os.path.exists(META_PATH)

print(f"BATCH_SIZE     = {BATCH_SIZE}")
print(f"MAX_NEW_TOKENS = {MAX_NEW_TOKENS}")
print(f"USE_MERGED     = {USE_MERGED}")
print(f"ADAPTER_DIR    = {ADAPTER_DIR}")

BATCH_SIZE     = 8
MAX_NEW_TOKENS = 1500
USE_MERGED     = False
ADAPTER_DIR    = ../outputs/qwen3_4b_pii_lora_tagged


## 2. 加载 test 集 + 元信息

In [3]:
with open(META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)
SYSTEM_PROMPT = meta["system_prompt"]
TARGET_LABELS = meta["target_labels"]

test_examples = []
with open(TEST_GT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        test_examples.append(json.loads(line))

print(f"test 样本数:     {len(test_examples)}")
print(f"target types:    {len(TARGET_LABELS)}")
pos_count = sum(1 for e in test_examples if e["spans"])
neg_count = len(test_examples) - pos_count
print(f"正样本(有 span): {pos_count}")
print(f"负样本(空 span): {neg_count}")

test 样本数:     3800
target types:    16
正样本(有 span): 1900
负样本(空 span): 1900


## 3. 加载模型和 tokenizer (关键:left padding)

In [4]:
if USE_MERGED:
    print(f"加载合并版: {MERGED_DIR}")
    model = AutoModelForCausalLM.from_pretrained(
        MERGED_DIR,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        attn_implementation="sdpa",
    )
    tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR, trust_remote_code=True)
else:
    print(f"加载 base: {BASE_MODEL}")
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        attn_implementation="sdpa",
    )
    print(f"挂载 adapter: {ADAPTER_DIR}")
    model = PeftModel.from_pretrained(base, ADAPTER_DIR)
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

# ★★★ 关键修复:decoder-only 模型推理必须 left padding ★★★
tokenizer.padding_side = "left"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()

print(f"\npadding_side:  {tokenizer.padding_side}")
print(f"pad_token:     {tokenizer.pad_token} (id={tokenizer.pad_token_id})")
print(f"eos_token:     {tokenizer.eos_token} (id={tokenizer.eos_token_id})")
if torch.cuda.is_available():
    print(f"GPU allocated: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")

`torch_dtype` is deprecated! Use `dtype` instead!


加载 base: ../../model/Qwen3-4B-Instruct-2507


Loading weights: 100%|████████████████████████████████████████████████████████████████| 398/398 [00:59<00:00,  6.66it/s]


挂载 adapter: ../outputs/qwen3_4b_pii_lora_tagged

padding_side:  left
pad_token:     <|endoftext|> (id=151643)
eos_token:     <|im_end|> (id=151645)
GPU allocated: 7.62 GB


## 4. 推理函数

In [5]:
def build_prompt(text: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Annotate PII in the following text:\n\n{text}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)

@torch.no_grad()
def infer_batch(texts: List[str]) -> List[str]:
    prompts = [build_prompt(t) for t in texts]
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LEN,
    ).to(model.device)

    out_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,                     # greedy
        pad_token_id=tokenizer.eos_token_id,
        # 不要传 temperature/top_p/top_k —— greedy 模式下它们被忽略还产生 warning
    )

    # 注意: left padding 下 inputs.input_ids.shape[1] = 整批最大长度(含 pad)
    # out_ids 前 seq_len 个 token 是 prompt(含 pad), 后面是生成的部分
    outputs = []
    seq_len = inputs.input_ids.shape[1]
    for i in range(len(prompts)):
        gen = out_ids[i][seq_len:]
        outputs.append(tokenizer.decode(gen, skip_special_tokens=True).strip())
    return outputs

# Sanity check: 跑一条看看输出格式对不对
print("Sanity check (第一条测试样本):")
sanity_input = test_examples[0]["text"]
print(f"INPUT  (前 200 字): {sanity_input[:200]}")
t0 = time.time()
sanity_out = infer_batch([sanity_input])
print(f"耗时: {time.time()-t0:.2f} s")
print(f"OUTPUT (前 400 字): {sanity_out[0][:400]}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Sanity check (第一条测试样本):
INPUT  (前 200 字): To the HR Director,

This correspondence is intended to place on record the particulars set out below.

A driver licence, TAS: 667104, was also presented as secondary identification. For completeness,
耗时: 12.53 s
OUTPUT (前 400 字): <think>

</think>

To the HR Director,

This correspondence is intended to place on record the particulars set out below.

A driver licence, <pii type="AU_DRIVERS_LICENCE">TAS: 667104</pii>, was also presented as secondary identification. For completeness, the tax records associated with this employment have been lodged under <pii type="AU_TFN">210 177 639</pii>.

Identity was established by trave


In [6]:
# ============================================================
# 快速评估 subset(100 条分层采样)
# ============================================================
import random
import time
import re
from collections import defaultdict

random.seed(42)

# 分层采样
pos_samples = [e for e in test_examples if e["spans"]]
neg_samples = [e for e in test_examples if not e["spans"]]
random.shuffle(pos_samples)
random.shuffle(neg_samples)
test_subset = pos_samples[:50] + neg_samples[:50]
random.shuffle(test_subset)
print(f"子集: {len(test_subset)} 条")

# 推理
predictions_sub = []
t0 = time.time()
for i in range(0, len(test_subset), BATCH_SIZE):
    batch = test_subset[i:i+BATCH_SIZE]
    texts = [e["text"] for e in batch]
    outs = infer_batch(texts)
    for ex, out in zip(batch, outs):
        predictions_sub.append({
            "text": ex["text"],
            "gt_spans": ex["spans"],
            "raw_output": out,
        })
    done = i + len(batch)
    if (i // BATCH_SIZE) % 3 == 0:
        rate = done / max(time.time()-t0, 0.01)
        print(f"  {done}/{len(test_subset)}  {rate:.2f} samples/s")

print(f"推理耗时: {(time.time()-t0)/60:.1f} min")

# 清理 <think> 块 + parse
_THINK = re.compile(r'<think>.*?</think>\s*', re.DOTALL)
for p in predictions_sub:
    p["raw_output"] = _THINK.sub('', p["raw_output"]).strip()
    p.update(eval_parse(p))

# 统计
n = len(predictions_sub)
n_parse_ok = sum(p["parse_ok"] for p in predictions_sub)
n_rt_ok = sum(p["round_trip_ok"] for p in predictions_sub)
print(f"\nParse OK:      {n_parse_ok}/{n}  ({n_parse_ok/n*100:.1f}%)")
print(f"Round-trip OK: {n_rt_ok}/{n}  ({n_rt_ok/n*100:.1f}%)")

# 快速 F1
m_exact = aggregate_metrics(predictions_sub, "exact")
m_partial = aggregate_metrics(predictions_sub, "partial", PARTIAL_IOU_THRESHOLD)

print(f"\n=== 子集评估结果 (100 条) ===")
print(f"Exact:   P={m_exact['overall']['precision']:.3f}  R={m_exact['overall']['recall']:.3f}  F1={m_exact['overall']['f1']:.3f}")
print(f"Partial: P={m_partial['overall']['precision']:.3f}  R={m_partial['overall']['recall']:.3f}  F1={m_partial['overall']['f1']:.3f}")

# 预测异常样本(给你快速看问题)
if n_rt_ok < n * 0.9:
    print("\n⚠️ Round-trip 失败样本:")
    for p in predictions_sub:
        if not p["round_trip_ok"]:
            print(f"  INPUT: {p['text'][:100]}...")
            print(f"  OUTPUT: {p['raw_output'][:200]}...")
            print()
            break  # 只打 1 条

子集: 100 条
  8/100  0.33 samples/s
  32/100  0.25 samples/s
  56/100  0.22 samples/s
  80/100  0.22 samples/s
  100/100  0.23 samples/s
推理耗时: 7.4 min


NameError: name 'eval_parse' is not defined

In [7]:
import re
from collections import defaultdict

_THINK = re.compile(r'<think>.*?</think>\s*', re.DOTALL)

def eval_parse(pred):
    # 清理 <think> 块
    cleaned = _THINK.sub('', pred['raw_output']).strip()
    try:
        plain, spans = parse_annotated(cleaned, strict=False)
    except Exception as e:
        return {
            "pred_spans": [],
            "parse_ok": False,
            "round_trip_ok": False,
            "parse_error_reason": str(e),
        }
    round_trip_ok = (plain == pred["text"])
    return {
        "pred_spans": spans,
        "parse_ok": True,
        "round_trip_ok": round_trip_ok,
        "parse_error_reason": None if round_trip_ok else "round_trip_mismatch",
    }


# 现在对已有的 predictions_sub 做 parse(不重新推理)
for p in predictions_sub:
    p.update(eval_parse(p))

# 统计
n = len(predictions_sub)
n_parse_ok = sum(p["parse_ok"] for p in predictions_sub)
n_rt_ok = sum(p["round_trip_ok"] for p in predictions_sub)
print(f"Parse OK:      {n_parse_ok}/{n}  ({n_parse_ok/n*100:.1f}%)")
print(f"Round-trip OK: {n_rt_ok}/{n}  ({n_rt_ok/n*100:.1f}%)")


# ---------- 下面是评估函数 ----------
def iou(a, b):
    inter = max(0, min(a["end"], b["end"]) - max(a["start"], b["start"]))
    union = max(a["end"], b["end"]) - min(a["start"], b["start"])
    return inter / union if union > 0 else 0.0

def match_spans(pred, gt, mode="exact", iou_threshold=0.5):
    cands = []
    for pi, p in enumerate(pred):
        for gi, g in enumerate(gt):
            if p["type"] != g["type"]:
                continue
            if mode == "exact":
                if p["start"] == g["start"] and p["end"] == g["end"]:
                    cands.append((1.0, pi, gi))
            elif mode == "partial":
                s = iou(p, g)
                if s >= iou_threshold:
                    cands.append((s, pi, gi))
    cands.sort(key=lambda x: -x[0])
    used_p, used_g = set(), set()
    matched = []
    for s, pi, gi in cands:
        if pi in used_p or gi in used_g:
            continue
        matched.append((pi, gi))
        used_p.add(pi)
        used_g.add(gi)
    unmatched_p = [i for i in range(len(pred)) if i not in used_p]
    unmatched_g = [i for i in range(len(gt)) if i not in used_g]
    return matched, unmatched_p, unmatched_g

def prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return {"precision": p, "recall": r, "f1": f, "tp": tp, "fp": fp, "fn": fn}

def aggregate(predictions, mode, iou_threshold=0.5):
    overall = {"tp": 0, "fp": 0, "fn": 0}
    per_type = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
    fp_list, fn_list = [], []
    for p in predictions:
        pred_spans = p["pred_spans"] if p["round_trip_ok"] else []
        gt_spans = p["gt_spans"]
        matched, up, ug = match_spans(pred_spans, gt_spans, mode, iou_threshold)
        for pi, gi in matched:
            t = gt_spans[gi]["type"]
            per_type[t]["tp"] += 1
            overall["tp"] += 1
        for pi in up:
            sp = pred_spans[pi]
            per_type[sp["type"]]["fp"] += 1
            overall["fp"] += 1
            if len(fp_list) < 10:
                fp_list.append({"text": p["text"][:150], "pred": sp, "gt": gt_spans})
        for gi in ug:
            sg = gt_spans[gi]
            per_type[sg["type"]]["fn"] += 1
            overall["fn"] += 1
            if len(fn_list) < 10:
                fn_list.append({"text": p["text"][:150], "missed": sg, "pred": pred_spans})
    return {
        "overall": prf(overall["tp"], overall["fp"], overall["fn"]),
        "per_type": {t: prf(v["tp"], v["fp"], v["fn"]) for t, v in per_type.items()},
        "fp_samples": fp_list,
        "fn_samples": fn_list,
    }


m_exact = aggregate(predictions_sub, "exact")
m_partial = aggregate(predictions_sub, "partial", 0.5)

print(f"\n=== 100 条子集评估结果 ===\n")
print(f"Exact match:")
e = m_exact["overall"]
print(f"  P={e['precision']:.4f}  R={e['recall']:.4f}  F1={e['f1']:.4f}  (TP={e['tp']} FP={e['fp']} FN={e['fn']})")
print()
print(f"Partial match (IoU >= 0.5):")
pa = m_partial["overall"]
print(f"  P={pa['precision']:.4f}  R={pa['recall']:.4f}  F1={pa['f1']:.4f}  (TP={pa['tp']} FP={pa['fp']} FN={pa['fn']})")

# Per-type
print(f"\n=== Per-type F1 (exact) ===")
print(f"{'Type':<25s} {'F1':>8s} {'P':>8s} {'R':>8s}  {'TP':>4s} {'FP':>4s} {'FN':>4s}")
for t in TARGET_LABELS:
    v = m_exact["per_type"].get(t)
    if v is None:
        print(f"{t:<25s} {'n/a':>8s}")
    else:
        print(f"{t:<25s} {v['f1']:>8.4f} {v['precision']:>8.4f} {v['recall']:>8.4f}  {v['tp']:>4d} {v['fp']:>4d} {v['fn']:>4d}")

# 看 1-2 个 FP/FN
if m_exact["fp_samples"]:
    print(f"\n=== 第一个 FP ===")
    s = m_exact["fp_samples"][0]
    print(f"text: {s['text']}")
    print(f"pred: {s['pred']}")

if m_exact["fn_samples"]:
    print(f"\n=== 第一个 FN ===")
    s = m_exact["fn_samples"][0]
    print(f"text: {s['text']}")
    print(f"missed: {s['missed']}")

Parse OK:      100/100  (100.0%)
Round-trip OK: 100/100  (100.0%)

=== 100 条子集评估结果 ===

Exact match:
  P=0.9648  R=0.9897  F1=0.9771  (TP=192 FP=7 FN=2)

Partial match (IoU >= 0.5):
  P=0.9698  R=0.9948  F1=0.9822  (TP=193 FP=6 FN=1)

=== Per-type F1 (exact) ===
Type                            F1        P        R    TP   FP   FN
PERSON                      1.0000   1.0000   1.0000    50    0    0
ADDRESS                     0.9677   0.9677   0.9677    30    1    1
DATE_OF_BIRTH               0.9434   0.9259   0.9615    25    2    1
EMAIL_ADDRESS               0.9630   0.9286   1.0000    13    1    0
AU_PHONE                    1.0000   1.0000   1.0000    17    0    0
AU_TFN                      1.0000   1.0000   1.0000     3    0    0
AU_PASSPORT                 1.0000   1.0000   1.0000     4    0    0
AU_DRIVERS_LICENCE          1.0000   1.0000   1.0000     3    0    0
STUDENT_ID                  1.0000   1.0000   1.0000     7    0    0
MEDICARE_NUMBER             1.0000   1.0000   1

## 5. 跑全量推理

预期 15-30 分钟。如果前 1-2 个 batch 的 samples/s > 2,说明修复生效。

In [6]:
predictions = []

t0 = time.time()
for i in range(0, len(test_examples), BATCH_SIZE):
    batch = test_examples[i:i+BATCH_SIZE]
    texts = [e["text"] for e in batch]
    outs = infer_batch(texts)
    for ex, out in zip(batch, outs):
        predictions.append({
            "text": ex["text"],
            "gt_spans": ex["spans"],
            "raw_output": out,
        })
    # 每 10 个 batch 打印一次进度
    if (i // BATCH_SIZE) % 10 == 0:
        done = i + len(batch)
        elapsed = time.time() - t0
        rate = done / max(elapsed, 0.01)
        eta = (len(test_examples) - done) / max(rate, 0.01)
        print(f"{done}/{len(test_examples)}  {rate:.2f} samples/s  ETA {eta/60:.1f} min")

total_min = (time.time() - t0) / 60
print(f"\n推理完成,总耗时 {total_min:.1f} min, 平均 {len(test_examples)/(total_min*60):.2f} samples/s")

# 保存 raw 输出
raw_path = os.path.join(REPORT_DIR, "predictions_raw.jsonl")
with open(raw_path, "w", encoding="utf-8") as f:
    for p in predictions:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")
print(f"raw 输出已保存: {raw_path}")

8/3800  0.43 samples/s  ETA 146.1 min


KeyboardInterrupt: 

## 6. Parse 模型输出

In [ ]:
import re

_THINK_PATTERN = re.compile(r'<think>.*?</think>\s*', re.DOTALL)
def eval_parse(pred):
    # 清理 <think>...</think> 块(Qwen3-Instruct 的固有行为)
    cleaned = _THINK_PATTERN.sub('', pred['raw_output']).strip()
    
    try:
        plain, spans = parse_annotated(cleaned, strict=False)
    except Exception as e:
        return {
            "pred_spans": [],
            "parse_ok": False,
            "round_trip_ok": False,
            "parse_error_reason": str(e),
        }
    round_trip_ok = (plain == pred["text"])
    return {
        "pred_spans": spans,
        "parse_ok": True,
        "round_trip_ok": round_trip_ok,
        "parse_error_reason": None if round_trip_ok else "round_trip_mismatch",
    }

for p in predictions:
    p.update(eval_parse(p))

n = len(predictions)
n_parse_ok = sum(p["parse_ok"] for p in predictions)
n_round_trip_ok = sum(p["round_trip_ok"] for p in predictions)

print(f"Parse OK:      {n_parse_ok}/{n}  ({n_parse_ok/n*100:.2f}%)")
print(f"Round-trip OK: {n_round_trip_ok}/{n}  ({n_round_trip_ok/n*100:.2f}%)")

## 7. Span-level 评估逻辑

In [ ]:
def iou(a, b):
    inter = max(0, min(a["end"], b["end"]) - max(a["start"], b["start"]))
    union = max(a["end"], b["end"]) - min(a["start"], b["start"])
    return inter / union if union > 0 else 0.0

def match_spans(pred, gt, mode="exact", iou_threshold=0.5):
    candidates = []
    for pi, p in enumerate(pred):
        for gi, g in enumerate(gt):
            if p["type"] != g["type"]:
                continue
            if mode == "exact":
                if p["start"] == g["start"] and p["end"] == g["end"]:
                    candidates.append((1.0, pi, gi))
            elif mode == "partial":
                score = iou(p, g)
                if score >= iou_threshold:
                    candidates.append((score, pi, gi))
    candidates.sort(key=lambda x: -x[0])
    used_p, used_g = set(), set()
    matched = []
    for score, pi, gi in candidates:
        if pi in used_p or gi in used_g:
            continue
        matched.append((pi, gi))
        used_p.add(pi)
        used_g.add(gi)
    unmatched_p = [i for i in range(len(pred)) if i not in used_p]
    unmatched_g = [i for i in range(len(gt)) if i not in used_g]
    return matched, unmatched_p, unmatched_g

def prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return {"precision": p, "recall": r, "f1": f, "tp": tp, "fp": fp, "fn": fn}

## 8. 聚合评估

In [ ]:
def aggregate_metrics(predictions, mode, iou_threshold=0.5):
    overall = {"tp": 0, "fp": 0, "fn": 0}
    per_type = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
    over_redaction_chars = 0
    under_redaction_cost = 0.0
    under_redaction_chars = 0
    fp_samples, fn_samples = [], []

    for p in predictions:
        pred_spans = p["pred_spans"] if p["round_trip_ok"] else []
        gt_spans = p["gt_spans"]
        matched, unmatched_p, unmatched_g = match_spans(
            pred_spans, gt_spans, mode=mode, iou_threshold=iou_threshold
        )
        for pi, gi in matched:
            t = gt_spans[gi]["type"]
            per_type[t]["tp"] += 1
            overall["tp"] += 1
        for pi in unmatched_p:
            sp = pred_spans[pi]
            per_type[sp["type"]]["fp"] += 1
            overall["fp"] += 1
            over_redaction_chars += sp["end"] - sp["start"]
            if len(fp_samples) < 30:
                fp_samples.append({
                    "text": p["text"][:200],
                    "pred_span": sp,
                    "gt_spans": gt_spans,
                })
        for gi in unmatched_g:
            sg = gt_spans[gi]
            per_type[sg["type"]]["fn"] += 1
            overall["fn"] += 1
            weight = HIGH_RISK_WEIGHT if sg["type"] in HIGH_RISK_TYPES else LOW_RISK_WEIGHT
            under_redaction_cost += weight
            under_redaction_chars += sg["end"] - sg["start"]
            if len(fn_samples) < 30:
                fn_samples.append({
                    "text": p["text"][:200],
                    "missed_gt": sg,
                    "pred_spans": pred_spans,
                })

    return {
        "mode": mode,
        "iou_threshold": iou_threshold if mode == "partial" else None,
        "overall": prf(overall["tp"], overall["fp"], overall["fn"]),
        "per_type": {t: prf(v["tp"], v["fp"], v["fn"]) for t, v in per_type.items()},
        "over_redaction_chars": over_redaction_chars,
        "under_redaction_cost": under_redaction_cost,
        "under_redaction_chars": under_redaction_chars,
        "fp_samples": fp_samples,
        "fn_samples": fn_samples,
    }

metrics_exact   = aggregate_metrics(predictions, "exact")
metrics_partial = aggregate_metrics(predictions, "partial", PARTIAL_IOU_THRESHOLD)
print("aggregate 完成")

## 9. 打印报告

In [ ]:
def print_metrics(m, title):
    print("=" * 72)
    print(title)
    print("=" * 72)
    o = m["overall"]
    print(f"Overall:  P={o['precision']:.4f}  R={o['recall']:.4f}  F1={o['f1']:.4f}")
    print(f"          TP={o['tp']}  FP={o['fp']}  FN={o['fn']}")
    print()
    print(f"{'Type':<25s} {'P':>8s} {'R':>8s} {'F1':>8s} {'TP':>6s} {'FP':>6s} {'FN':>6s}")
    print("-" * 72)
    for t in TARGET_LABELS:
        v = m["per_type"].get(t)
        if v is None:
            print(f"{t:<25s} {'(no data)':>8s}")
        else:
            print(f"{t:<25s} {v['precision']:>8.4f} {v['recall']:>8.4f} {v['f1']:>8.4f} "
                  f"{v['tp']:>6d} {v['fp']:>6d} {v['fn']:>6d}")
    print()
    print(f"Over-redaction (FP chars):   {m['over_redaction_chars']}")
    print(f"Under-redaction cost:        {m['under_redaction_cost']:.1f}")
    print(f"Under-redaction (FN chars):  {m['under_redaction_chars']}")

print_metrics(metrics_exact, "Exact match (start/end/type 全对)")
print()
print_metrics(metrics_partial, f"Partial match (IoU >= {PARTIAL_IOU_THRESHOLD} 且 type 对)")

## 10. Parse / Round-trip 失败样例

In [ ]:
round_trip_failures = [p for p in predictions if not p["round_trip_ok"]]
print(f"Round-trip 失败的样本: {len(round_trip_failures)} / {len(predictions)}")
if round_trip_failures:
    print("\n前 3 个样例:")
    for i, p in enumerate(round_trip_failures[:3]):
        print(f"\n--- sample {i+1} ---")
        print(f"input       : {p['text'][:200]}...")
        print(f"raw_output  : {p['raw_output'][:300]}...")
        print(f"error       : {p['parse_error_reason']}")

## 11. FP / FN 样例

In [ ]:
print("=== FP 前 5 个 (exact match) ===")
for i, s in enumerate(metrics_exact["fp_samples"][:5]):
    print(f"\n--- FP {i+1} ---")
    print(f"text     : {s['text']}")
    print(f"pred     : {s['pred_span']}")
    print(f"gt spans : {[(g['start'], g['end'], g['type']) for g in s['gt_spans']]}")

print("\n\n=== FN 前 5 个 (exact match) ===")
for i, s in enumerate(metrics_exact["fn_samples"][:5]):
    print(f"\n--- FN {i+1} ---")
    print(f"text      : {s['text']}")
    print(f"missed gt : {s['missed_gt']}")
    print(f"pred spans: {[(p['start'], p['end'], p['type']) for p in s['pred_spans']]}")

## 12. 保存报告

In [ ]:
def clean_for_json(m):
    return {
        "mode": m["mode"],
        "iou_threshold": m["iou_threshold"],
        "overall": m["overall"],
        "per_type": m["per_type"],
        "over_redaction_chars": m["over_redaction_chars"],
        "under_redaction_cost": m["under_redaction_cost"],
        "under_redaction_chars": m["under_redaction_chars"],
    }

report = {
    "model": {
        "base": BASE_MODEL,
        "adapter": ADAPTER_DIR,
        "use_merged": USE_MERGED,
    },
    "test_size": len(test_examples),
    "parse_ok_rate": n_parse_ok / n,
    "round_trip_ok_rate": n_round_trip_ok / n,
    "exact": clean_for_json(metrics_exact),
    "partial": clean_for_json(metrics_partial),
    "high_risk_types": sorted(HIGH_RISK_TYPES),
    "high_risk_weight": HIGH_RISK_WEIGHT,
    "inference_config": {
        "batch_size": BATCH_SIZE,
        "max_new_tokens": MAX_NEW_TOKENS,
        "max_input_len": MAX_INPUT_LEN,
        "padding_side": tokenizer.padding_side,
    },
}

report_path = os.path.join(REPORT_DIR, "metrics.json")
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
print(f"saved: {report_path}")

full_path = os.path.join(REPORT_DIR, "metrics_with_samples.json")
with open(full_path, "w", encoding="utf-8") as f:
    json.dump({"exact": metrics_exact, "partial": metrics_partial},
              f, ensure_ascii=False, indent=2)
print(f"saved: {full_path}")

## 13. Redaction 演示

In [ ]:
demo = [p for p in predictions if p["round_trip_ok"] and p["pred_spans"]][:3]
for d in demo:
    print("=" * 72)
    print("INPUT:")
    print(d["text"][:400])
    print("\nREDACTED:")
    redacted = apply_redaction(d["text"], d["pred_spans"], mode="mask")
    print(redacted[:400])
    print()